# 🏎️ F1 Pit Stop Prediction — Notebook 1: Exploratory Data Analysis
**Kaggle Playground Series S6E5**  
Goal: Predict whether a driver will pit on the **next lap** (binary classification, AUC metric)

## 0. Setup

In [2]:
find /Users/clairedebadts/kaggle/Playground_Series/S6/kagglecs6_predictingF1PitStops -type f | head -30

SyntaxError: invalid syntax (1156988780.py, line 1)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data')

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '../data/train.csv'

## 1. First Look

In [ ]:
train.head(10)

In [ ]:
train.info()

In [ ]:
train.describe(include='all')

## 2. Missing Values & Dtypes

In [ ]:
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df.missing_count > 0])
print('\nNo missing values!' if missing_df.missing_count.sum() == 0 else '')

In [ ]:
# Columns shared between train and test (target excluded)
train_cols = set(train.columns) - {'PitNextLap'}
test_cols  = set(test.columns)
print('In train only:', train_cols - test_cols)
print('In test only: ', test_cols - train_cols)

## 3. Target Distribution

In [ ]:
target_counts = train['PitNextLap'].value_counts()
target_pct    = train['PitNextLap'].value_counts(normalize=True) * 100

print('Target distribution:')
print(pd.DataFrame({'count': target_counts, 'pct': target_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No Pit (0)', 'Pit (1)'], target_counts.values,
       color=['#4C72B0', '#DD8452'])
for i, (v, p) in enumerate(zip(target_counts.values, target_pct.values)):
    ax.text(i, v + 1000, f'{p:.1f}%', ha='center', fontsize=11)
ax.set_title('PitNextLap — Class Imbalance')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Target rate by year — check for yearly inconsistency (flagged in discussions)
train.groupby('Year')['PitNextLap'].agg(['mean', 'count']).rename(
    columns={'mean': 'pit_rate', 'count': 'rows'}
).assign(pit_rate=lambda df: (df.pit_rate * 100).round(2))

## 4. Categorical Features

In [ ]:
cat_cols = ['Compound', 'Race', 'Driver', 'Year']

for col in cat_cols:
    n_unique = train[col].nunique()
    top5 = train[col].value_counts().head(5)
    pit_rate = train.groupby(col)['PitNextLap'].mean().sort_values(ascending=False).head(5)
    print(f'\n--- {col} ({n_unique} unique values) ---')
    print('Top 5 by count:')
    print(top5.to_string())
    print('Top 5 by pit rate:')
    print((pit_rate * 100).round(2).to_string())

In [ ]:
# Compound pit rate — key strategic signal
compound_stats = train.groupby('Compound').agg(
    pit_rate=('PitNextLap', 'mean'),
    count=('PitNextLap', 'count'),
    median_tyre_life=('TyreLife', 'median')
).sort_values('pit_rate', ascending=False)
compound_stats['pit_rate'] = (compound_stats['pit_rate'] * 100).round(2)
print(compound_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compound distribution
compound_counts = train['Compound'].value_counts()
axes[0].bar(compound_counts.index, compound_counts.values, color=sns.color_palette('Set2'))
axes[0].set_title('Compound — Race Share')
axes[0].set_ylabel('Lap count')

# Pit rate by compound
pit_by_compound = train.groupby('Compound')['PitNextLap'].mean().sort_values(ascending=False)
axes[1].bar(pit_by_compound.index, pit_by_compound.values * 100, color=sns.color_palette('Set2'))
axes[1].set_title('Pit Rate by Compound (%)')
axes[1].set_ylabel('Pit rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

## 5. Numerical Features

In [ ]:
num_cols = ['LapNumber', 'Stint', 'TyreLife', 'Position']
train[num_cols].describe()

In [ ]:
# Distribution of numerical features split by PitNextLap
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), num_cols):
    for val, label, color in [(0, 'No Pit', '#4C72B0'), (1, 'Pit', '#DD8452')]:
        subset = train[train['PitNextLap'] == val][col]
        ax.hist(subset, bins=50, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(f'{col} — by PitNextLap')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Feature Distributions by Target', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 6. TyreLife — The Central Signal

In [ ]:
# Pit rate as a function of TyreLife — binned
train['TyreLife_bin'] = pd.cut(train['TyreLife'], bins=range(0, 80, 5))
tyre_pit = train.groupby('TyreLife_bin', observed=True)['PitNextLap'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(tyre_pit)), tyre_pit.values * 100, color='#DD8452')
ax.set_xticks(range(len(tyre_pit)))
ax.set_xticklabels([str(b) for b in tyre_pit.index], rotation=45, ha='right')
ax.set_title('Pit Rate by TyreLife (binned, 5-lap windows)')
ax.set_ylabel('Pit rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

train.drop(columns='TyreLife_bin', inplace=True)

In [ ]:
# TyreLife pit rate split by Compound
fig, axes = plt.subplots(1, train['Compound'].nunique(), figsize=(18, 4), sharey=True)

for ax, compound in zip(axes, sorted(train['Compound'].unique())):
    subset = train[train['Compound'] == compound].copy()
    subset['TyreLife_bin'] = pd.cut(subset['TyreLife'], bins=range(0, 80, 3))
    pit_rate = subset.groupby('TyreLife_bin', observed=True)['PitNextLap'].mean()
    ax.bar(range(len(pit_rate)), pit_rate.values * 100, color='#4C72B0', alpha=0.8)
    ax.set_title(f'{compound}')
    ax.set_xlabel('TyreLife (3-lap bins)')
    if ax == axes[0]:
        ax.set_ylabel('Pit rate (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.suptitle('Pit Rate by TyreLife per Compound', fontsize=13)
plt.tight_layout()
plt.show()

## 7. LapNumber & Race Context

In [ ]:
# Total race length per race (max LapNumber)
race_lengths = train.groupby(['Year', 'Race'])['LapNumber'].max().reset_index()
race_lengths.columns = ['Year', 'Race', 'TotalLaps']
print(f'Race length range: {race_lengths.TotalLaps.min()} – {race_lengths.TotalLaps.max()} laps')
print(race_lengths.sort_values('TotalLaps', ascending=False).head(10).to_string(index=False))

In [ ]:
# Merge total laps back and compute LapProgress (0–1)
train = train.merge(race_lengths, on=['Year', 'Race'], how='left')
train['LapProgress'] = train['LapNumber'] / train['TotalLaps']
train['LapsRemaining'] = train['TotalLaps'] - train['LapNumber']

# Pit rate by race progress
train['LapProgress_bin'] = pd.cut(train['LapProgress'], bins=20)
prog_pit = train.groupby('LapProgress_bin', observed=True)['PitNextLap'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(prog_pit)), prog_pit.values * 100, color='#55A868')
ax.set_xticks(range(len(prog_pit)))
ax.set_xticklabels([f'{b.mid:.0%}' for b in prog_pit.index], rotation=45, ha='right')
ax.set_title('Pit Rate by Race Progress (%)')
ax.set_ylabel('Pit rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

train.drop(columns=['LapProgress_bin'], inplace=True)

In [ ]:
# Nobody pits in the final few laps — confirm the cliff
print('Pit rate for last 3 laps of race:')
print(train[train['LapsRemaining'] <= 3].groupby('LapsRemaining')['PitNextLap'].mean())

print('\nPit rate for laps 1–3 of race:')
print(train[train['LapNumber'] <= 3].groupby('LapNumber')['PitNextLap'].mean())

## 8. Position & Stint

In [ ]:
# Pit rate by position group
train['PositionGroup'] = pd.cut(train['Position'], bins=[0,5,10,15,20], labels=['P1-5','P6-10','P11-15','P16-20'])
pos_pit = train.groupby('PositionGroup', observed=True)['PitNextLap'].mean()
print('Pit rate by position group:')
print((pos_pit * 100).round(2))
train.drop(columns='PositionGroup', inplace=True)

In [ ]:
# Pit rate by stint number
stint_pit = train.groupby('Stint')['PitNextLap'].agg(['mean', 'count'])
stint_pit['mean'] = (stint_pit['mean'] * 100).round(2)
print('Pit rate by stint:')
print(stint_pit)

## 9. PitStop Column vs PitNextLap

In [ ]:
# PitStop = 1 means this lap was a pit lap
# PitNextLap = 1 means the NEXT lap is a pit
# These should be consistent: PitStop[lap N] == PitNextLap[lap N-1]
print('PitStop vs PitNextLap crosstab:')
print(pd.crosstab(train['PitStop'], train['PitNextLap'], 
                  rownames=['PitStop (this lap)'], 
                  colnames=['PitNextLap (next lap)'],
                  normalize='all').round(4))

In [ ]:
# Verify temporal consistency within a driver/race session
# Sort by driver, race, year, lap and check shift
train_sorted = train.sort_values(['Year', 'Race', 'Driver', 'LapNumber']).copy()
train_sorted['PitStop_next'] = train_sorted.groupby(
    ['Year', 'Race', 'Driver'])['PitStop'].shift(-1)

# When PitNextLap==1, PitStop on the next row should be 1
match_pct = (train_sorted['PitStop_next'] == train_sorted['PitNextLap']).mean()
print(f'PitNextLap ↔ next-row PitStop alignment: {match_pct:.2%}')

## 10. Correlation & Feature Importance Proxy

In [ ]:
# Point-biserial correlations with target
from scipy.stats import pointbiserialr

feature_cols = ['LapNumber', 'Stint', 'TyreLife', 'Position', 
                'LapProgress', 'LapsRemaining', 'TotalLaps']

corrs = {}
for col in feature_cols:
    r, p = pointbiserialr(train[col].dropna(), train.loc[train[col].notna(), 'PitNextLap'])
    corrs[col] = {'correlation': round(r, 4), 'p_value': p}

corr_df = pd.DataFrame(corrs).T.sort_values('correlation', key=abs, ascending=False)
print(corr_df)

In [ ]:
# Quick LightGBM importance check (no tuning, just signal ranking)
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

quick = train.copy()
for col in ['Compound', 'Race', 'Driver']:
    quick[col] = LabelEncoder().fit_transform(quick[col].astype(str))

features = ['LapNumber', 'Stint', 'TyreLife', 'Position',
            'Compound', 'Race', 'Driver', 'Year',
            'LapProgress', 'LapsRemaining', 'TotalLaps']

clf = lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1)
clf.fit(quick[features], quick['PitNextLap'])

imp = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind='barh', ax=ax, color='#4C72B0')
ax.invert_yaxis()
ax.set_title('Quick LightGBM Feature Importance (signal proxy)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 11. Train vs Test Distribution Check

In [ ]:
# Check for distribution shift between train and test
num_check = ['LapNumber', 'TyreLife', 'Position', 'Stint']

for col in num_check:
    t_mean = train[col].mean()
    te_mean = test[col].mean()
    t_std = train[col].std()
    te_std = test[col].std()
    print(f'{col:15s} | train mean={t_mean:.2f} std={t_std:.2f} | test mean={te_mean:.2f} std={te_std:.2f}')

In [ ]:
# Compound split in train vs test
print('Compound — train vs test share:')
comp_train = train['Compound'].value_counts(normalize=True).rename('train')
comp_test  = test['Compound'].value_counts(normalize=True).rename('test')
print(pd.concat([comp_train, comp_test], axis=1).round(3))

In [ ]:
# Year split in train vs test
print('Year — train vs test:')
print(pd.DataFrame({
    'train': train['Year'].value_counts().sort_index(),
    'test':  test['Year'].value_counts().sort_index()
}))

## 12. EDA Summary & Key Findings

| Finding | Implication |
|---------|-------------|
| ~14% positive rate (class imbalance) | Use `scale_pos_weight` or `is_unbalance=True` in LightGBM |
| TyreLife is strongest predictor | Engineer `TyreLifeRatio` per compound in notebook 02 |
| Pits cluster mid-race (LapProgress ~30–70%) | `LapProgress` and `LapsRemaining` are high-signal features |
| Near-zero pit rate in final 3 laps | Hard rule or feature flag `is_final_laps` |
| PitStop (current lap) aligns with PitNextLap | Check alignment — PitStop may leak; verify in notebook 02 |
| Compound affects typical pit window | `TyreLife / median_compound_life` normalisation needed |
| Year-level differences in pit rate | Stratify CV by Year |
| 887 unique drivers | High-cardinality — use target encoding, not one-hot |
| 26 unique races | Target encode or embed |

**Next: `02_cleaning_features.ipynb`** — leakage check, feature engineering, encoding strategy